In [3]:
%load_ext autoreload
%autoreload 2

In [38]:
import math
import os

import duckdb
from sindex.metrics.batch_jobs import create_topics_table
from sindex.metrics.citations import merge_citations_from_files_fast
from sindex.metrics.topics import enhance_topics, restructure_topics_ndjson

## Citations

In [5]:
mdc_citations = r"D:\may-2026-data\citations\mdc\mdc_citations_datacite.ndjson"
oa_citations = r"D:\may-2026-data\citations\openalex\oa_citations.ndjson"
dc_citations = r"D:\may-2026-data\citations\datacite\dc_citations.ndjson"
dc_citations_previous_datasets = (
    r"D:\may-2026-data\citations\datacite-previous\dc_citations.ndjson"
)
mdc_citations_emdb = r"D:\may-2026-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [
    mdc_citations,
    oa_citations,
    dc_citations,
    dc_citations_previous_datasets,
    mdc_citations_emdb,
]
output_file = r"D:\may-2026-data\citations\citations.ndjson"

In [6]:
merge_citations_from_files_fast(citation_files, output_file)

Starting merge of 5 valid files...
Finished processing 972,293 records. Unique: 956,456
Writing to D:\may-2026-data\citations\citations.ndjson...
Done!


## Mentions

## Research Fields / Topics

### Enhance fair scores from OpenAlex with subfiled, field, and domain

In [11]:
input_topics = r"D:\may-2026-data\topics\topics-oa\doi_topics_oa.ndjson"
mapping_file = (
    r"D:\combined-data\external\openalex-topics\openalex_topic_mapping_table.csv"
)
output_topics = r"D:\may-2026-data\topics\topics-oa\doi_topics_oa_enhanced.ndjson"

In [12]:
enhance_topics(input_topics, mapping_file, output_topics)

Starting  enhancement
Loading CSV
CSV loaded. 4,516 topics indexed.
Processing lines
Lines processed: 4,700,000 | Matches: 4,700,000

Done!
Total lines: 4,749,534
Total matches: 4,749,534
Saved to: D:\may-2026-data\topics\topics-oa\doi_topics_oa_enhanced.ndjson


### Standardize and group our topics assignment from our custom model

In [17]:
input_ndjson_path = r"D:\may-2026-data\topics\topics-custom-model\topics-files"
output_path = r"D:\may-2026-data\topics\topics-custom-model\topics_custom_model.ndjson"

In [18]:
restructure_topics_ndjson(input_ndjson_path, output_path)

Processing: 214 out of 214 files...
Processing complete
Total files processed: 214
Total lines in output: 21,275,684


### Load topics in a db for picking easily picking best topics oa vs custom model

In [19]:
topics_db = r"D:\may-2026-data\topics\topics.duckdb"

In [26]:
oa_file = r"D:\may-2026-data\topics\topics-oa\doi_topics_oa_enhanced.ndjson"
con = duckdb.connect(topics_db)
con.execute(
    f"""
        CREATE OR REPLACE TABLE topics_oa AS 
        SELECT * FROM read_json_auto('{oa_file}', ignore_errors=true);
    """
)
row_count = con.execute("SELECT COUNT(*) FROM topics_oa").fetchone()[0]
print(f"Total rows: {row_count}")
display(con.execute("SELECT * FROM topics_oa LIMIT 3").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 4749534


,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10.26037/yareta:e3zggd5vx5hnnbsrgtbnmuxngm,T13949,Belt and Road Initiative,0.0603,openalex,2002,Economics and Econometrics,20,"Economics, Econometrics and Finance",2,Social Sciences,China-Pakistan Economic Corridor; Sustainable ...,This cluster of papers focuses on the impact o...,https://en.wikipedia.org/wiki/China%E2%80%93Pa...
1,10.26037/yareta:ef2h6hlu5zc3zhw6fiuywsj54m,T10184,Plant Molecular Biology Research,0.3592,openalex,1110,Plant Science,11,Agricultural and Biological Sciences,1,Life Sciences,Plant; Development; MicroRNA; Auxin; Gene Expr...,This cluster of papers focuses on the molecula...,https://en.wikipedia.org/wiki/Plant_development
2,10.26037/yareta:mzfaokbfr5e6hoxnowsgjt62fe,T11128,Transition Metal Oxide Nanomaterials,0.1697,openalex,2507,Polymers and Plastics,25,Materials Science,3,Physical Sciences,Electrochromic; Tungsten Oxide; Vanadium Oxide...,This cluster of papers focuses on advanced mat...,https://en.wikipedia.org/wiki/Smart_window


In [28]:
custom_file = r"D:\may-2026-data\topics\topics-custom-model\topics_custom_model.ndjson"
con = duckdb.connect(topics_db)
con.execute(
    f"""
        CREATE OR REPLACE TABLE topics_custom_model AS 
        SELECT * FROM read_json_auto('{custom_file}', ignore_errors=true);
    """
)
row_count = con.execute("SELECT COUNT(*) FROM topics_custom_model").fetchone()[0]
print(f"Total rows: {row_count}")
display(con.execute("SELECT * FROM topics_custom_model LIMIT 3").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 21275684


,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,EMD-55311,T11767,Poxvirus research and outbreaks,0.5717,custom_model,2406,Virology,24,Immunology and Microbiology,1,Life Sciences,None,None,None
1,EMD-56837,T11441,Organoboron and organosilicon chemistry,0.3225,custom_model,1605,Organic Chemistry,16,Chemistry,3,Physical Sciences,None,None,None
2,EMD-55758,T11597,"Neutrophil, Myeloperoxidase and Oxidative Mech...",0.4253,custom_model,2403,Immunology,24,Immunology and Microbiology,1,Life Sciences,None,None,None


##### Create final topics table (OpenAlex if exist and score >0.5 else custom if score> than OA or OA not exist)

In [36]:
create_topics_table(topics_db)

Creating final 'topics' table with score comparison logic


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Table created in 41.87 seconds.
Final 'topics' table contains 21,275,684 rows.

Sample of rows where Custom Model won:
                    dataset_id        source   score
0  10.57451/lhd.a.nbi.148849.1  custom_model  0.4541
1  10.57451/lhd.a.nbi.148852.1  custom_model  0.4523
2  10.57451/lhd.a.nbi.148853.1  custom_model  0.4538
3  10.57451/lhd.a.nbi.148854.1  custom_model  0.4571
4  10.57451/lhd.a.nbi.148855.1  custom_model  0.4483


#### Export topics for our cloud database

In [40]:
topics_split_folder = r"D:\may-2026-data\topics\topics-split"
lines_per_file = 500_000

os.makedirs(topics_split_folder, exist_ok=True)

con = duckdb.connect(topics_db)
total_rows = con.execute("SELECT COUNT(*) FROM topics").fetchone()[0]
num_files = math.ceil(total_rows / lines_per_file)
for i in range(num_files):
    offset = i * lines_per_file
    chunk_path = os.path.join(topics_split_folder, f"topics_part_{i+1}.ndjson")
    con.execute(
        f"""
        COPY (
            SELECT * FROM topics
            LIMIT {lines_per_file} OFFSET {offset}
        )
        TO '{chunk_path}'
        (FORMAT JSON, ARRAY FALSE);
    """
    )
    rows_written = min((i + 1) * lines_per_file, total_rows)
    print(f"\rFile {i+1}/{num_files} | {rows_written:,}/{total_rows:,} rows", end="", flush=True)
con.close()
print(f"\nFinished! Wrote {num_files} files, {total_rows:,} records total.")

File 43/43 | 21,275,684/21,275,684 rows
Finished! Wrote 43 files, 21,275,684 records total.
